HW 8 Linear Regression and Least Squares problem

In [7]:
import numpy as np
import pandas as pd

In [9]:
walmart_sales = pd.read_csv("Walmart_Sales.csv")
walmart_sales.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106


In [10]:
walmart_sales['Store']

0        1
1        1
2        1
3        1
4        1
        ..
6430    45
6431    45
6432    45
6433    45
6434    45
Name: Store, Length: 6435, dtype: int64

I want to create a hyperplane that fits the data and represents how the weekly sales are impacted by the other features. In my mind i'd like to accumulate the weekly sales of all of the stores together and then see how to find the optimal weight vector of features W and the bias term B. This will provide a national wide analysis of the impact features have on the total weekly sales. So first I want to start by grouping the weekly sales of all the stores combined. This will remove the impact that each particular store could have on weekly sales, but will allow for a more general estimation of the impact of other features on the data. I will sum the sales for that week while averaging out the other features. I will also drop the Temperature feature as I think that since it is associated with the region of the store, averaging out the temperature of so many regions may not be useful. My intuition on this is that even though temperature probably has an impact on an individual store's sales, averaging out such a wide spread of national temperatures on a particular week, will be a detriment to the analysis of the impact of the other features. 

In [11]:
walmart_sales = walmart_sales.drop("Temperature", axis=1)

In [12]:
weekly_sales_of_all_stores = walmart_sales.groupby("Date").agg({
    "Weekly_Sales": "sum",
    "Holiday_Flag": "mean",
    "Fuel_Price": "mean",
    "CPI": "mean",
    "Unemployment": "mean"
})

weekly_sales_of_all_stores.head()

,Weekly_Sales,Holiday_Flag,Fuel_Price,CPI,Unemployment
Date,,,,,
01-04-2011,43458991.19,0.0,3.602356,170.725418,8.150133
01-06-2012,48281649.72,0.0,3.750822,175.603188,7.419533
01-07-2011,47578519.50,0.0,3.675978,171.395827,8.097489
01-10-2010,42239875.87,0.0,2.734333,168.354706,8.475289
02-03-2012,46861034.97,0.0,3.696022,174.921137,7.508333


# ![function](image.png)

now I can apply the formula above to the data

In [23]:
# Minimize φ(w, b) = Σ_i (y_i - (w^T x_i + b))^2 

feature_cols = ["Holiday_Flag", "Fuel_Price", "CPI", "Unemployment"]

y = weekly_sales_of_all_stores["Weekly_Sales"].to_numpy(dtype=np.float64, copy=True)
X = weekly_sales_of_all_stores[feature_cols].to_numpy(dtype=np.float64, copy=True)

# Augmented design: each row is [x_i^T, 1] so predictions are X_aug @ [w; b]
X_aug = np.column_stack([X, np.ones(len(y), dtype=np.float64)])

# Optimal (w, b) solves the normal equations; lstsq returns the minimizer of φ
theta, residuals, rank, s = np.linalg.lstsq(X_aug, y, rcond=None)
w_opt = theta[:-1]
b_opt = float(theta[-1])
y_hat = X_aug @ theta
phi_opt = np.sum((y - y_hat) ** 2)  # minimum value of φ


print("Optimal w (per feature):")
for name, wi in zip(feature_cols, w_opt):
    print(f"  {name:16s}  {wi:+,.2f}")
print(f"\nOptimal bias b: {b_opt:+,.2f}")
print(f"\nMinimum φ(w, b) = SSE: {phi_opt:,.2f}")
rmse = np.sqrt(np.mean((y - y_hat) ** 2))
print(f"\nRoot Mean Squared Error (RMSE): {rmse:,.2f}")



Optimal w (per feature):
  Holiday_Flag      +2,785,019.20
  Fuel_Price        -5,242,303.67
  CPI               +2,519,193.97
  Unemployment      +12,277,275.72

Optimal bias b: -465,921,537.85

Minimum φ(w, b) = SSE: 3,857,386,843,438,453.00

Root Mean Squared Error (RMSE): 5,193,720.55


Surprisingly these results imply that unemployment percentage has a high correlation with weekly sales with a positive association, meaning as unemployment percentages increase, this data predicts higher sales. Holidays and CPI also tend to show a strong positive linear association with sales. While fuel_prices impact is negatively associated quite heavily. It's important to note that each feature has it's own unit of measurement and thus the interpretation of the results and their impact depends on that.

Fuel price: stated in dollar per gallon
Holiday Flag: Wether its a holiday or not
CPI: Consumer Price Index based on index values
Unemployment: Percentage of unemployment rate.

The optimal bias is a really big negative number which makes sense since the weekly sales of all walmart stores is a really big $$$ number. Essentially this b helps as a starting constant insinuating the impact of all other factor we are not accounting for in this data. To me it helps to think that the features will never be at 0 (except holiday flag) in real world scenarios, so the b helps best fit the correlations of the other features.